In [1]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

PyTorch: 2.5.1
CUDA available: True
GPU: NVIDIA RTX A6000
VRAM total: 47.4 GB


Load Qwen 2.5 BF16

In [2]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # use only GPU 0 (A6000 has 47.4GB, enough for 7B)

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, time

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,  # BF16: same exponent range as FP32, recommended for Qwen2.5
    device_map="cuda:0"
)
load_time = time.time() - t0

print(f"Load time: {load_time:.1f}s")
print(f"Model device: {next(model.parameters()).device}")
print(f"Allocated VRAM : {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"Reserved  VRAM : {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Load time: 4.3s
Model device: cuda:0
Allocated VRAM : 14.19 GB
Reserved  VRAM : 14.21 GB


In [3]:
prompt = "Explain what is model quantization in one paragraph."
messages = [{"role": "user", "content": prompt}]

# chat template for Qwen2.5
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

# warmup
with torch.no_grad():
    _ = model.generate(**inputs, max_new_tokens=10, do_sample=False)

torch.cuda.synchronize()

# generate and measure time
t0 = time.time()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
torch.cuda.synchronize()
elapsed = time.time() - t0

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
n_tokens = outputs.shape[1] - inputs.input_ids.shape[1]

print(f"Number of tokens generated: {n_tokens}")
print(f"Elapsed time: {elapsed:.2f}s")
print(f"Throughput: {n_tokens / elapsed:.1f} tokens/s")
print(f"\nResponse:\n{response}")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Number of tokens generated: 89
Elapsed time: 2.26s
Throughput: 39.3 tokens/s

Response:
Model quantization is a technique used to reduce the precision of the weights and activations in neural network models, typically from 32-bit floating-point numbers to lower bit depths like 8-bit integers or even binary values, which significantly reduces the memory footprint and computational requirements of the model without significantly compromising its performance. This process helps in making deep learning models more efficient for deployment on devices with limited resources, such as mobile phones or embedded systems.


In [4]:
from transformers import BitsAndBytesConfig

# free BF16 model before loading 8-bit
del model
torch.cuda.empty_cache()

bnb_8bit = BitsAndBytesConfig(load_in_8bit=True)

t0 = time.time()
model_8bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_8bit,
    device_map="cuda:0"
)
load_time_8bit = time.time() - t0

print(f"Load time      : {load_time_8bit:.1f}s")
print(f"Allocated VRAM : {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"Reserved  VRAM : {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Load time      : 14.9s
Allocated VRAM : 8.12 GB
Reserved  VRAM : 8.63 GB


In [ ]:
# 8-bit inference
inputs_8bit = tokenizer(text, return_tensors="pt").to(model_8bit.device)

# warmup
with torch.no_grad():
    _ = model_8bit.generate(**inputs_8bit, max_new_tokens=10, do_sample=False)

torch.cuda.synchronize()

# generate and measure time
t0 = time.time()
with torch.no_grad():
    outputs_8bit = model_8bit.generate(**inputs_8bit, max_new_tokens=200, do_sample=False)
torch.cuda.synchronize()
elapsed_8bit = time.time() - t0

response_8bit = tokenizer.decode(outputs_8bit[0][inputs_8bit.input_ids.shape[1]:], skip_special_tokens=True)
n_tokens_8bit = outputs_8bit.shape[1] - inputs_8bit.input_ids.shape[1]

print(f"Number of tokens generated: {n_tokens_8bit}")
print(f"Elapsed time: {elapsed_8bit:.2f}s")
print(f"Throughput: {n_tokens_8bit / elapsed_8bit:.1f} tokens/s")
print(f"\nResponse:\n{response_8bit}")


In [ ]:
# free 8-bit model before loading 4-bit
del model_8bit
torch.cuda.empty_cache()

bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",               # NormalFloat4: optimized for normally-distributed weights
    bnb_4bit_compute_dtype=torch.bfloat16,   # dequantize to BF16 for compute
    bnb_4bit_use_double_quant=True,          # double quantization: also quantize the scale factor (~0.4 bit/param extra saving)
)

t0 = time.time()
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_4bit,
    device_map="cuda:0"
)
load_time_4bit = time.time() - t0

print(f"Load time      : {load_time_4bit:.1f}s")
print(f"Allocated VRAM : {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"Reserved  VRAM : {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")


In [ ]:
# 4-bit NF4 inference
inputs_4bit = tokenizer(text, return_tensors="pt").to(model_4bit.device)

# warmup
with torch.no_grad():
    _ = model_4bit.generate(**inputs_4bit, max_new_tokens=10, do_sample=False)

torch.cuda.synchronize()

# generate and measure time
t0 = time.time()
with torch.no_grad():
    outputs_4bit = model_4bit.generate(**inputs_4bit, max_new_tokens=200, do_sample=False)
torch.cuda.synchronize()
elapsed_4bit = time.time() - t0

response_4bit = tokenizer.decode(outputs_4bit[0][inputs_4bit.input_ids.shape[1]:], skip_special_tokens=True)
n_tokens_4bit = outputs_4bit.shape[1] - inputs_4bit.input_ids.shape[1]

print(f"Number of tokens generated: {n_tokens_4bit}")
print(f"Elapsed time: {elapsed_4bit:.2f}s")
print(f"Throughput: {n_tokens_4bit / elapsed_4bit:.1f} tokens/s")
print(f"\nResponse:\n{response_4bit}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results = {
    "Precision":  ["BF16",       "INT8",          "NF4 4-bit"   ],
    "VRAM (GB)":  [14.19,        8.12,            None          ],  # fill after running
    "Throughput": [n_tokens/elapsed, n_tokens_8bit/elapsed_8bit, n_tokens_4bit/elapsed_4bit],
    "Load time":  [load_time,    load_time_8bit,  load_time_4bit],
}

df = pd.DataFrame(results)
print(df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, col, color in zip(axes, ["VRAM (GB)", "Throughput", "Load time"],
                                  ["steelblue", "seagreen", "salmon"]):
    vals = df[col].astype(float)
    ax.bar(df["Precision"], vals, color=color)
    ax.set_title(col)
    ax.set_ylabel(col)
    for i, v in enumerate(vals):
        ax.text(i, v * 1.01, f"{v:.1f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("benchmark_results.png", dpi=150)
plt.show()
print("Saved to benchmark_results.png")
